[Reference](https://medium.com/timescale/finding-the-best-open-source-embedding-model-for-rag-929d1656d331)

In [1]:
def create_vectorizer(embedding_model, embeddings_dimensions):
    embeddings_view_name = f"{'essays'}{'_'}{embedding_model.replace('-','_')}{'_'}{'embeddings'}"

    with connect_db() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT ai.create_vectorizer(
                'essays'::regclass,
                destination => %s,
                embedding => ai.embedding_ollama(%s, %s),
                chunking => ai.chunking_recursive_character_text_splitter('text', 512, 50),
                formatting => ai.formatting_python_template('title: $title $chunk')
            );
            """, (embeddings_view_name, embedding_model, embeddings_dimensions, )
            )

In [3]:
with connect_db() as conn:
   with conn.cursor() as cur:
        # Load Paul Graham's essays dataset into the 'essays' table
        cur.execute("""
            SELECT ai.load_dataset(
                    'sgoel9/paul_graham_essays',
                    table_name => 'essays',
                    if_table_exists => 'append');
        """)

In [5]:
EMBEDDING_MODELS = [
    {'name':'mxbai-embed-large', 'dimensions': 1024},
    {'name':'nomic-embed-text','dimensions': 768},
    {'name':'bge-m3','dimensions': 1024},
]

for model in EMBEDDING_MODELS:
    create_vectorizer(model['name'], model['dimensions'])

In [6]:
with connect_db() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT * FROM ai.vectorizer_status;")

        for row in cur.fetchall():
print(f"Vectorizer ID: {row[0]}, Embedding Table: {row[2]}, Pending Items: {row[4]}")

In [7]:
def generate_questions_by_question_type(chunk, question_type, num_questions):
    prompts = {
        'short': "Generate {count} short, simple questions about this text. Questions should be direct, under 10 words",
        'long': "Generate {count} detailed, comprehensive questions about this text. Include specific details:",
        'direct': "Generate {count} questions that directly ask about explicit information in this text",
        'implied': "Generate {count} questions that require understanding context and implications of the text:",
        'unclear': "Generate {count} vague, ambiguous questions about the general topic of this text:"
    }

    prompt = prompts[question_type].format(count=num_questions) + f"\n\nText: {chunk}"

    system_instructions = """
        Generate different types of questions about the given text following the prompt provided.
        Each question must be on a new line. Do not include empty lines or blank questions.
    """
    with connect_db() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT ai.ollama_generate(
                    'llama3.2',
                    %s,
                    system_prompt=>%s,
                    host=>%s
                )->>'response';
            """,(prompt, system_instructions, OLLAMA_HOST))
            generated_questions = [q.strip() for q in cur.fetchone()[0].split("\n") if q.strip()]
            print(f"Number of questions generated for {question_type}: {len(generated_questions)}")
            return generated_questions